# Notebook 01 — Data Understanding and Problem Framing
**Project:** Mining Process Quality Prediction — Minsur Analytics Challenge  
**Level:** 1 (mandatory)  
**Target:** `% Silica Concentrate`  
**Author:** Analytics Candidate  

---

## Business Context

In the Minsur flotation process, the quality of the final iron ore concentrate
is measured by the **percentage of silica** (`% Silica Concentrate`) — a lower value
means purer iron concentrate. Laboratory quality measurements arrive with a
**significant delay** (up to several hours), which prevents real-time operational
decisions.

The goal of this project is to **predict % Silica Concentrate in real time**
using continuous sensor data (ore feed quality, reagent flows, flotation column
parameters), enabling operators to react proactively to quality excursions.

---

## Why Temporal Integrity Matters

> A random train/test split on time-series data introduces **data leakage**:
> the model would train on observations from March to predict January, effectively
> seeing the future. Evaluation metrics would be over-optimistic and the model
> would fail in production. We strictly respect the chronological ordering of
> all splits throughout this project.

In [ ]:
# ---------------------------------------------------------------------------
# 0. Environment setup — add project root to sys.path so src/ is importable
# ---------------------------------------------------------------------------
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import warnings

warnings.filterwarnings('ignore')



import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

import matplotlib.dates as mdates

import seaborn as sns



from src.config import CFG, PROJECT_ROOT

from src.data_preprocessing import (

    load_raw_data,

    normalize_column_names,

    parse_and_sort_datetime,

    remove_duplicates,

    align_to_modeling_frequency,

    data_quality_report,

    outlier_summary,

    sampling_frequency_report,

    run_preprocessing_pipeline,

)

from src.evaluate import compute_metrics



sns.set_style('whitegrid')

plt.rcParams['figure.dpi'] = 120

FIGURES_PATH = Path(CFG['paths']['reports_figures'])



TARGET = CFG['target']

print(f"Target: {TARGET}")


---
## 1. Load Dataset

The raw CSV is expected in `data/raw/`. If not yet downloaded, run the Kaggle
download cell below (requires `kagglehub` and valid Kaggle credentials).

In [ ]:
# ---------------------------------------------------------------------------

# Download via kagglehub if the file does not already exist

# (aligned with src.data_preprocessing auto-download behavior)

# ---------------------------------------------------------------------------

import shutil

import sys

import subprocess



try:

    import kagglehub

except ModuleNotFoundError:

    print('kagglehub not found in current kernel. Installing...')

    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'kagglehub'])

    import kagglehub



raw_dir = Path(CFG['paths']['data_raw'])

raw_file = raw_dir / CFG['data']['raw_filename']

raw_dir.mkdir(parents=True, exist_ok=True)



if not raw_file.exists():

    print('Downloading dataset from Kaggle via kagglehub.dataset_download...')

    dataset_dir = Path(

        kagglehub.dataset_download('edumagalhaes/quality-prediction-in-a-mining-process')

    )



    csv_candidates = sorted(dataset_dir.rglob('*.csv'))

    if not csv_candidates:

        raise FileNotFoundError(f'No CSV found in downloaded dataset path: {dataset_dir}')



    source_csv = next(

        (p for p in csv_candidates if 'mining' in p.name.lower() or 'flotation' in p.name.lower()),

        csv_candidates[0],

    )

    shutil.copy2(source_csv, raw_file)

    print(f'Copied: {source_csv} -> {raw_file}')

else:

    print(f'File already exists: {raw_file}')


In [ ]:
# Load raw (uncleaned) to inspect as-is
df_raw = load_raw_data(CFG)
df_raw = normalize_column_names(df_raw)
print(f'Shape: {df_raw.shape}')
df_raw.head(3)

---
## 2. Datetime Parsing and Temporal Ordering

In [ ]:
df = parse_and_sort_datetime(df_raw.copy(), CFG)
df = remove_duplicates(df)

print(f'Date range: {df.index.min()} → {df.index.max()}')
print(f'Shape after deduplication: {df.shape}')
df.head(3)

---
## 3. Target and Feature Identification

In [ ]:
# Identify target and candidate predictors
lab_cols = CFG['feature_groups']['lab_measurements']  # delayed — NOT predictors

feature_candidates = [c for c in df.columns if c not in lab_cols]

print(f'Target: {TARGET}')
print(f'\nLab measurement columns (delayed — excluded from features):')
print(lab_cols)
print(f'\nCandidate predictors ({len(feature_candidates)}):')
for c in feature_candidates:
    print(f'  {c}')

> **Feature availability assumptions**

| Variable group | Examples | Availability at inference time | Decision |
|---|---|---|---|
| Delayed lab measurements | `% Iron Concentrate`, `% Silica Concentrate` | No (delayed lab) | Exclude as current predictors; only past lags of target allowed in sensitivity |
| Feed composition | `% Iron Feed`, `% Silica Feed` | Assumption-dependent (site-specific instrumentation) | Train main model with feed + run sensitivity without feed |
| Online operational sensors | `Starch Flow`, `Amina Flow`, `Ore Pulp pH`, `Ore Pulp Density`, flotation air/levels | Yes (assumed online) | Keep as core predictors |

**Leakage guard:** the target `% Silica Concentrate` is never used as current predictor; if lagged target is used, it is strictly `shift >= 1` (past-only).

---
## 4. Data Quality Analysis

In [ ]:
report = data_quality_report(df)
print('=== Data Quality Report ===')
report

In [ ]:
# Missing values summary
missing = report[['n_missing', 'pct_missing']].sort_values('n_missing', ascending=False)
print('Missing values per column:')
print(missing[missing['n_missing'] > 0])
print(f'\nTotal missing: {df.isnull().sum().sum()}')

In [ ]:
# Outlier summary (IQR × 3)
out_summary = outlier_summary(df)
print('Outlier summary (IQR × 3 factor):')
print(out_summary.head(15))

In [ ]:
# Sampling frequency analysis (raw effective frequency)

freq_report_raw = sampling_frequency_report(df)

print('Inter-sample time deltas BEFORE modeling alignment (minutes):')

print(freq_report_raw.head(10))



# Explicit modeling alignment frequency

modeling_rule = CFG['data'].get('modeling_frequency', '1h')

modeling_agg = CFG['data'].get('modeling_aggregation', 'mean')

df_model = align_to_modeling_frequency(df, rule=modeling_rule, agg=modeling_agg)



freq_report_model = sampling_frequency_report(df_model)

print('\nInter-sample time deltas AFTER modeling alignment (minutes):')

print(freq_report_model.head(10))

print(f'\nRows: raw={len(df):,} | aligned_for_modeling={len(df_model):,}')


### Frecuencia temporal: original vs target vs modelado

No asumimos una frecuencia fija por defecto. Se calcula explícitamente con:
`df.index.to_series().diff().value_counts()` antes y después de la alineación temporal.

| Nivel | Qué representa | Cómo se calcula en notebook |
|---|---|---|
| Frecuencia original del dataset | Ritmo efectivo de registro en el CSV crudo | `freq_report_raw` |
| Frecuencia efectiva del target | Ritmo al que `% Silica Concentrate` cambia realmente (laboratorio, con retardo) | inspección de cambios + repeticiones del target |
| Frecuencia final de modelado | Grano temporal que usamos para entrenamiento | `align_to_modeling_frequency(rule='1h')` + `freq_report_model` |

### Sobre “duplicados” y reducción de filas

La reducción fuerte de filas **no** se interpreta como `drop_duplicates()` ciego. Se interpreta como:
1. Eliminación de duplicados exactos (higiene básica).
2. **Alineación temporal explícita** a frecuencia de modelado (`1h`, agregación `mean`) para que cada fila represente un intervalo operacional comparable con la disponibilidad del target.

Esto evita mezclar múltiples snapshots de alta frecuencia dentro del mismo horizonte de decisión y hace defendible el split temporal train/val/test.

### Mixed Sampling Frequencies — Implications

The dataset contains variables with **different update frequencies**:

| Variable group | Approx. update frequency |
|---|---|
| Sensor data (flows, levels, pH, density) | Higher-frequency process stream |
| Lab measurements (`% Iron`, `% Silica Concentrate`) | Lower frequency with reporting delay |

**Implications:**
1. Repeated target values and delayed lab timestamps can inflate naive correlation.
2. Leakage risk increases if delayed lab variables are used as contemporaneous predictors.
3. Explicit temporal alignment and past-only feature generation are required for defensible modeling.

---
## 5. Visual Analysis

In [ ]:
# ── 5a. Target distribution ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df[TARGET].dropna(), bins=50, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title(f'Distribution of {TARGET}', fontsize=12)
axes[0].set_xlabel(TARGET)
axes[0].set_ylabel('Count')

axes[1].boxplot(df[TARGET].dropna(), vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[1].set_title(f'Boxplot — {TARGET}', fontsize=12)
axes[1].set_ylabel(TARGET)
axes[1].set_xticks([])

plt.tight_layout()
plt.savefig(FIGURES_PATH / 'target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(df[TARGET].describe().round(4))

### Hallazgo corto — Target distribution

La variable `% Silica Concentrate` está **sesgada a la derecha**: la mayor densidad cae entre ~1.0 y 2.5, con mediana cercana a **2.0** y una cola hacia valores altos (hasta ~5.53). Esto sugiere episodios menos frecuentes de sílice elevada (peor calidad), por lo que en modelado conviene vigilar errores grandes (RMSE y análisis de residuos) además del MAE.

In [ ]:
# ── 5b. Temporal evolution of the target ─────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df.index, df[TARGET], lw=0.5, color='steelblue', alpha=0.7)
ax.set_title(f'Temporal Evolution — {TARGET}', fontsize=13)
ax.set_xlabel('Date')
ax.set_ylabel(TARGET)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.tight_layout()
plt.savefig(FIGURES_PATH / 'target_temporal.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 5c. Correlation matrix ────────────────────────────────────────────────
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()

# Focus on top correlated features with target
target_corr = corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
top_cols = target_corr.head(15).index.tolist() + [TARGET]

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr.loc[top_cols, top_cols], dtype=bool))
sns.heatmap(
    corr.loc[top_cols, top_cols],
    annot=True, fmt='.2f', mask=mask, cmap='RdBu_r',
    center=0, linewidths=0.5, ax=ax, annot_kws={'fontsize': 8}
)
ax.set_title(f'Correlation Matrix — Top features vs {TARGET}', fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_PATH / 'correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 10 correlations with target:')
print(target_corr.head(10).round(3))

### Conclusiones de la mtriz de correlación

> Estas correlaciones son **lineales** y **no implican causalidad**. Se usan para entender señales preliminares y riesgos de modelado.

1. **Driver más fuerte del target:** `% Iron Concentrate` muestra una correlación alta y negativa con `% Silica Concentrate` (≈ -0.80).
   - Es coherente metalúrgicamente (más hierro en concentrado suele implicar menos sílice).
   - **Pero esta variable es de laboratorio y llega con retraso**, por lo que no debe usarse como feature directa en inferencia online (riesgo de leakage).

2. **Señales operativas con impacto moderado en sílice:**
   - `Flotation Column 03 Air Flow` y `Flotation Column 01 Air Flow` aparecen con correlación negativa moderada (≈ -0.22 / -0.21).
   - `Amina Flow` aparece con correlación positiva moderada (≈ +0.16), sugiriendo comportamiento no lineal o interacción con otras variables.
   - `Ore Pulp pH` y niveles de columnas muestran efectos débiles a moderados, útiles para feature engineering.

3. **Multicolinealidad fuerte entre variables de proceso:**
   - Flujos de aire entre columnas están altamente correlacionados entre sí (ej. ≈ 0.85–0.95).
   - `% Iron Feed` y `% Silica Feed` tienen correlación muy alta en magnitud (≈ -0.97).
   - Implicancia: para modelos lineales conviene regularización (Ridge) y para árboles revisar importancia/SHAP para evitar interpretaciones engañosas.

4. **Implicancia para la siguiente etapa (Level 2):**
   - Priorizar variables operativas disponibles en tiempo real.
   - Excluir variables de laboratorio contemporáneas.
   - Incorporar lags, rolling means/std e interacciones para capturar dinámica temporal y no depender solo de correlación instantánea.

---
## 6. Metric Definitions and Baseline Model

### Metrics

| Metric | Formula | Interpretation |
|---|---|---|
| **MAE** | $\frac{1}{n}\sum|y_i - \hat{y}_i|$ | Average absolute error in % Silica units |
| **RMSE** | $\sqrt{\frac{1}{n}\sum(y_i - \hat{y}_i)^2}$ | Penalises large errors; relevant for quality excursions |
| **R²** | $1 - \frac{SS_{res}}{SS_{tot}}$ | Proportion of variance explained; 1.0 = perfect |

### Baseline Strategy
The simplest possible model: **predict the mean of the training target for every observation**.  
Any useful model must beat this baseline; otherwise it provides no value over the current
status quo of 'just use the average'.

In [ ]:
# ---------------------------------------------------------------------------
# Temporal split for baseline evaluation — NO random split allowed
# ---------------------------------------------------------------------------
from src.feature_engineering import temporal_split

# Use the cleaned full DataFrame (no feature engineering yet)
df_clean = df.dropna(subset=[TARGET])

train_df, val_df, test_df = temporal_split(
    df_clean,
    train_ratio=CFG['split']['train_ratio'],
    val_ratio=CFG['split']['val_ratio'],
)

In [ ]:
# Baseline: predict training mean on all splits
train_mean = train_df[TARGET].mean()

baseline_results = {}
for split_name, split_df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    y_true = split_df[TARGET]
    y_pred = np.full(len(y_true), train_mean)
    metrics = compute_metrics(y_true, y_pred)
    baseline_results[split_name] = metrics
    print(f'Baseline ({split_name}): {metrics}')

print(f'\nBaseline prediction (mean of training target): {train_mean:.4f}')

In [ ]:
# Save baseline metrics for comparison in notebook 02
import json
metrics_path = Path(CFG['paths']['reports_metrics'])
with open(metrics_path / 'baseline_metrics.json', 'w') as f:
    json.dump(baseline_results, f, indent=2)
print(f'Baseline metrics saved to: {metrics_path / "baseline_metrics.json"}')

---
## 7. Save Cleaned Data for Downstream Notebooks

In [ ]:
# Run the full preprocessing pipeline and persist to data/interim/

import sys

import subprocess



# Ensure parquet engine exists in the active notebook kernel

try:

    import pyarrow  # noqa: F401

except ModuleNotFoundError:

    print('pyarrow not found in current kernel. Installing...')

    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pyarrow'])



df_cleaned = run_preprocessing_pipeline(CFG)

print(f'Cleaned data shape: {df_cleaned.shape}')

df_cleaned.head(3)


---
## Summary — Level 1 Checklist

| Item | Status |
|---|---|
| Dataset loaded from `data/raw/` | ✅ |
| Datetime column parsed and sorted | ✅ |
| Target identified: `% Silica Concentrate` | ✅ |
| Candidate predictors identified | ✅ |
| Missing values analysed | ✅ |
| Duplicates exactos tratados | ✅ |
| Alineación temporal explícita para modelado (`1h`) | ✅ |
| Frecuencia real analizada con `diff().value_counts()` | ✅ |
| Mixed sampling frequencies explained | ✅ |
| Feature availability assumptions documented | ✅ |
| Visual analysis (distribution, temporal, correlation, missing) | ✅ |
| Metric definitions (MAE, RMSE, R²) | ✅ |
| Baseline model implemented and evaluated | ✅ |
| No random split — temporal integrity preserved | ✅ |
| Leakage risks explicitly documented | ✅ |

### Conclusiones ejecutivas (Level 1)

- **Target**: se modela `% Silica Concentrate` como regresión temporal para anticipar calidad antes del laboratorio.
- **Frecuencia**: se distingue entre frecuencia original del dataset, frecuencia efectiva del target y frecuencia final de modelado (`1h`, agregación explícita).
- **Data quality**: la reducción de filas no se justifica por `drop_duplicates()` ciego, sino por alineación temporal.
- **Riesgos de leakage**: variables de laboratorio se excluyen como predictoras contemporáneas; target sólo puede entrar como lag pasado en análisis de sensibilidad.
- **Baseline**: queda establecido como piso mínimo (media del train) para exigir mejora real de modelos no triviales.